# 03 — Programación asincrónica: sync vs async

**Level 0 — Fundamentos Software & IA**

Simulamos consultas a modelos LLM (con demoras) para comparar:

- **Secuencial**: una consulta espera a la anterior (suma de demoras)
- **Concurrente**: todas arrancan juntas (demora máxima)
- **Errores** y **timeouts**: cómo se manejan en tareas concurrentes

In [1]:
import asyncio
import time
from typing import Any


async def consultar_modelo(modelo: str, prompt: str, demora: float = 1.0) -> dict[str, Any]:
    """Simula una consulta a un modelo LLM."""
    print(f"   [INICIO] Consultando {modelo}...")
    await asyncio.sleep(demora)
    respuesta = f"Respuesta simulada de {modelo} para: {prompt[:20]}..."
    print(f"   [FIN]    {modelo} completo ({demora}s)")
    return {
        "modelo": modelo,
        "prompt": prompt,
        "respuesta": respuesta,
        "latencia": demora,
    }

## Secuencial: uno por vez

In [2]:
async def ejecutar_secuencial() -> list[dict[str, Any]]:
    print("\n  --- SECUENCIAL (uno por vez) ---")
    resultados: list[dict[str, Any]] = []
    resultados.append(await consultar_modelo("gpt-4", "Explica Python async", 1.5))
    resultados.append(await consultar_modelo("claude-3", "Escribe un poema", 1.0))
    resultados.append(await consultar_modelo("llama3", "Traduce al ingles", 0.5))
    return resultados

inicio = time.time()
resultados_sec = await ejecutar_secuencial()
tiempo_sec = time.time() - inicio
for r in resultados_sec:
    print(f"   {r['modelo']}: {r['respuesta'][:40]}...")
print(f"\n   Tiempo secuencial: {tiempo_sec:.2f}s")


  --- SECUENCIAL (uno por vez) ---
   [INICIO] Consultando gpt-4...


   [FIN]    gpt-4 completo (1.5s)
   [INICIO] Consultando claude-3...


   [FIN]    claude-3 completo (1.0s)
   [INICIO] Consultando llama3...


   [FIN]    llama3 completo (0.5s)
   gpt-4: Respuesta simulada de gpt-4 para: Explic...
   claude-3: Respuesta simulada de claude-3 para: Esc...
   llama3: Respuesta simulada de llama3 para: Tradu...

   Tiempo secuencial: 3.00s


## Concurrente: todas a la vez

In [3]:
async def ejecutar_concurrente() -> list[dict[str, Any]]:
    print("\n  --- CONCURRENTE (todas a la vez) ---")
    resultados = await asyncio.gather(
        consultar_modelo("gpt-4", "Explica Python async", 1.5),
        consultar_modelo("claude-3", "Escribe un poema", 1.0),
        consultar_modelo("llama3", "Traduce al ingles", 0.5),
    )
    return resultados

inicio = time.time()
resultados_con = await ejecutar_concurrente()
tiempo_con = time.time() - inicio
for r in resultados_con:
    print(f"   {r['modelo']}: {r['respuesta'][:40]}...")
print(f"\n   Tiempo concurrente: {tiempo_con:.2f}s")


  --- CONCURRENTE (todas a la vez) ---
   [INICIO] Consultando gpt-4...
   [INICIO] Consultando claude-3...
   [INICIO] Consultando llama3...


   [FIN]    llama3 completo (0.5s)


   [FIN]    claude-3 completo (1.0s)


   [FIN]    gpt-4 completo (1.5s)
   gpt-4: Respuesta simulada de gpt-4 para: Explic...
   claude-3: Respuesta simulada de claude-3 para: Esc...
   llama3: Respuesta simulada de llama3 para: Tradu...

   Tiempo concurrente: 1.50s


## Comparación

In [4]:
print("\n  --- COMPARACION ---")
print(f"   Secuencial:  {tiempo_sec:.2f}s (suma de demoras)")
print(f"   Concurrente: {tiempo_con:.2f}s (demora mas larga)")
print(f"   Diferencia:  {tiempo_sec / tiempo_con:.1f}x mas rapido")


  --- COMPARACION ---
   Secuencial:  3.00s (suma de demoras)
   Concurrente: 1.50s (demora mas larga)
   Diferencia:  2.0x mas rapido


## Manejo de errores y timeouts

In [5]:
async def consultar_con_error(modelo: str, debe_fallar: bool = False) -> dict[str, Any]:
    print(f"   [INICIO] {modelo}...")
    await asyncio.sleep(0.5)
    if debe_fallar:
        raise RuntimeError(f"Error simulado en {modelo}")
    return {"modelo": modelo, "status": "ok"}

async def ejecutar_con_errores() -> None:
    print("\n  --- MANEJO DE ERRORES ---")
    resultados = await asyncio.gather(
        consultar_con_error("modelo-A", debe_fallar=False),
        consultar_con_error("modelo-B", debe_fallar=True),
        consultar_con_error("modelo-C", debe_fallar=False),
        return_exceptions=True,
    )
    for i, r in enumerate(resultados):
        if isinstance(r, Exception):
            print(f"   [ERROR] Tarea {i} fallo: {r}")
        else:
            print(f"   [OK]    Tarea {i}: {r['modelo']} - {r['status']}")

async def tarea_lenta() -> str:
    print("   [INICIO] Tarea lenta...")
    await asyncio.sleep(10)
    return "Termine!"

async def ejecutar_con_timeout() -> None:
    print("\n  --- TIMEOUT ---")
    try:
        resultado = await asyncio.wait_for(tarea_lenta(), timeout=2.0)
        print(f"   Resultado: {resultado}")
    except asyncio.TimeoutError:
        print("   [TIMEOUT] La tarea lenta no respondio en 2 segundos")

await ejecutar_con_errores()
await ejecutar_con_timeout()


  --- MANEJO DE ERRORES ---
   [INICIO] modelo-A...
   [INICIO] modelo-B...
   [INICIO] modelo-C...


   [OK]    Tarea 0: modelo-A - ok
   [ERROR] Tarea 1 fallo: Error simulado en modelo-B
   [OK]    Tarea 2: modelo-C - ok

  --- TIMEOUT ---
   [INICIO] Tarea lenta...


   [TIMEOUT] La tarea lenta no respondio en 2 segundos


## Conclusión

- **Secuencial**: tiempo = suma de demoras
- **Concurrente**: tiempo = demora más larga (con `asyncio.gather`)
- Los **errores** se capturan con `return_exceptions=True`
- Los **timeouts** con `asyncio.wait_for`